In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion # 把我之前写错的那个也删掉
!rm -rf master.zip

# 2. 克隆仓库 (注意：这次名字是对的！)
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字（注意带 's'）
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
import math
from google.colab import files
from PIL import Image, ImageOps

# === 核心工具函数 ===

def create_circular_mask(size=256):
    """创建圆形遮罩：圆内是1，圆外是0"""
    Y, X = np.ogrid[:size, :size]
    center = size / 2 - 0.5
    dist_from_center = np.sqrt((X - center)**2 + (Y - center)**2)
    mask = dist_from_center <= center # 只有圆内是 True
    return torch.from_numpy(mask).float().to(device).unsqueeze(0)

def rotate_tensor(image, angle_deg):
    """旋转图片并保持圆形裁剪"""
    # BILINEAR 插值保证旋转流畅
    rotated = TF.rotate(image, angle_deg, interpolation=TF.InterpolationMode.BILINEAR)
    # 再次乘遮罩，防止旋转时边缘出现伪影
    return rotated * CIRCULAR_MASK

def apply_composite(layer_a, layer_b):
    """
    合成逻辑：模拟两张幻灯片叠加
    公式：(A * B * 亮度增强).tanh() -> 压制到 0-1 之间
    """
    brightness = 3.0 
    # 核心叠图公式
    combined = (layer_a * layer_b * brightness)
    # 限制范围并切圆
    return combined.clamp(0, 99).tanh() * CIRCULAR_MASK

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    
    # 初始化遮罩 (必须在有了 device 之后)
    CIRCULAR_MASK = create_circular_mask(256)
    print("模型加载完毕！")
else:
    print("模型已存在，跳过加载。")
    # 确保遮罩也在正确的设备上
    if 'CIRCULAR_MASK' not in dir():
        CIRCULAR_MASK = create_circular_mask(256)

In [ ]:
print(">>> 请点击下方按钮上传你的'谜底'图片 (例如写着 LOVE 的黑底白字图) <<<")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    
    # 1. 读取并缩放到 256x256
    target_pil = Image.open(filename).convert('RGB')
    target_pil = ImageOps.fit(target_pil, (256, 256), method=Image.Resampling.LANCZOS)
    
    # 2. 转为 Tensor
    raw_target_tensor = TF.to_tensor(target_pil).to(device)
    
    # 3. 【关键步骤】立刻应用圆形遮罩
    # 这确保了我们只把 LOVE 图片的中间圆形部分当作目标
    # 四个角的像素会被强制变为黑色 (0)，不参与 Loss 计算
    secret_target_tensor = raw_target_tensor * CIRCULAR_MASK
    
    print("\n目标图片处理完毕！我们只会去匹配圆圈内的内容（如下图）：")
    rp.display_image(rp.as_numpy_image(secret_target_tensor))
else:
    print("❌ 未上传图片，请重新运行此块！")

In [ ]:
# === 🎮 游戏参数 ===
SECRET_ANGLE = 45        # 谜题答案：旋转多少度显示 LOVE？(可以是负数)
GUIDANCE_STRENGTH = 2500 # 隐写强度：越大字越清晰，越小越艺术 (推荐 2000-3000)

# === 🎨 画面描述 ===
# 第一张图（基底）：彩色玻璃窗风格
prompt_base = "An intricate circular stained glass window design, colorful, detailed, geometry"
# 第二张图（旋转层）：曼达拉花纹
prompt_decoder = "A beautiful circular mandala pattern, fractal art, symmetry, magic circle"

negative_prompt = "blur, text, letters, low quality, ugly, distortion, square edges"

# === 初始化可训练图像 ===
# 创建两个生成器
image_maker = lambda: LearnableImageFourier(height=256, width=256, hidden_dim=256, num_features=256).to(device)

raw_layer_base = image_maker()
raw_layer_decoder = image_maker()

# 定义获取图像的函数（每次获取都会自动切圆）
get_layer_base = lambda: raw_layer_base() * CIRCULAR_MASK
get_layer_decoder = lambda: raw_layer_decoder() * CIRCULAR_MASK

# 准备标签和优化器
label_base = NegativeLabel(prompt_base, negative_prompt)
label_decoder = NegativeLabel(prompt_decoder, negative_prompt)

# 我们同时优化两张图
params = chain(raw_layer_base.parameters(), raw_layer_decoder.parameters())
optim = torch.optim.SGD(params, lr=1e-4)

print(f"初始化完成。设置解谜角度为: {SECRET_ANGLE}°")

In [ ]:
NUM_ITER = 3000           # 训练次数，不够可以改大
DISPLAY_INTERVAL = 200    # 每200次显示一次

model_sd.max_step = 980
model_sd.min_step = 20

display_eta = rp.eta(NUM_ITER, title='Training Status')

print(f"🚀 开始训练... 目标：旋转 {SECRET_ANGLE} 度时显现目标图。")

history = []

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)

        # --- A. 正常的 SD 训练 (让图片好看) ---
        # 轮流训练 Base 图和 Decoder 图
        if iter_num % 2 == 0:
            curr_image = get_layer_decoder()
            curr_label = label_decoder
        else:
            curr_image = get_layer_base()
            curr_label = label_base

        # 计算让图片像 Prompt 的 Loss
        _ = model_sd.train_step(
            curr_label.embedding,
            curr_image[None],
            noise_coef=0.1,
            guidance_scale=60
        )

        # --- B. 关键步骤：隐写引导 (Steganography Loss) ---
        
        # 1. 拿两张图
        img_base = get_layer_base()
        img_decoder = get_layer_decoder()
        
        # 2. 模拟解谜操作：旋转 Decoder 层
        img_rotated = rotate_tensor(img_decoder, SECRET_ANGLE)
        
        # 3. 叠加两张图
        composite_result = apply_composite(img_base, img_rotated)
        
        # 4. 计算和目标图的差距 (MSE Loss)
        # 这里的 secret_target_tensor 已经被切过圆了，所以只对比圆形区域
        loss_secret = torch.mean((composite_result - secret_target_tensor)**2) * GUIDANCE_STRENGTH
        
        # 5. 反向传播
        loss_secret.backward()

        # --- C. 显示进度 ---
        with torch.no_grad():
            if iter_num % DISPLAY_INTERVAL == 0:
                from IPython.display import clear_output
                clear_output(wait=True)
                
                # 准备展示数据
                base_np = rp.as_numpy_image(img_base)
                dec_np = rp.as_numpy_image(img_decoder)
                
                # 错误角度预览 (比如 0 度叠加) -> 应该是乱的
                wrong_composite = apply_composite(img_base, rotate_tensor(img_decoder, 0))
                wrong_np = rp.as_numpy_image(wrong_composite)
                
                # 正确角度预览 (解谜) -> 应该显示 LOVE
                correct_composite = composite_result 
                correct_np = rp.as_numpy_image(correct_composite)

                # 拼图: 上面是两张原图，下面是 错误叠加 vs 正确叠加
                row1 = np.hstack([base_np, dec_np])
                row2 = np.hstack([wrong_np, correct_np])
                full_grid = np.vstack([row1, row2])
                
                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"上排: 图1 (基底) | 图2 (旋转层)")
                print(f"下排: 0°重叠 (乱码) | {SECRET_ANGLE}°重叠 (你的目标图!)")
                
                rp.display_image(full_grid)

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("用户手动停止训练。")

In [ ]:
print("==== 最终成果展示 ====")

print("1. 打印给玩家的第一张图 (Base Layer):")
rp.display_image(rp.as_numpy_image(get_layer_base()))

print("2. 打印给玩家的第二张图 (Decoder Layer):")
rp.display_image(rp.as_numpy_image(get_layer_decoder()))

print(f"3. 玩家把图2旋转 {SECRET_ANGLE} 度放在图1上 (解谜成功):")
solved_image = apply_composite(get_layer_base(), rotate_tensor(get_layer_decoder(), SECRET_ANGLE))
rp.display_image(rp.as_numpy_image(solved_image))

print("4. 玩家如果没有转对角度 (例如转了 180 度 - 失败):")
wrong_image = apply_composite(get_layer_base(), rotate_tensor(get_layer_decoder(), 180))
rp.display_image(rp.as_numpy_image(wrong_image))